# 15 — Data augmentation e avaliação robusta em visão

**EP · Data Science Knowledge Base | Deep Learning | Notebook 15/20**


**Pré-requisitos:** Python, NumPy, fundamentos de aprendizado supervisionado; JupyterLab; TensorFlow/Keras apenas nos laboratórios indicados.

**Escopo de validação:** os exemplos básicos podem ser reproduzidos sem TensorFlow; laboratórios opcionais precisam de validação local independente.

---

**Deep Learning · Capítulo de estudo independente**

> Material didático autoral · estudo independente · exemplos locais e laboratórios opcionais

**Referência curricular:** `material histórico de referência`. O material original foi preservado; este capítulo é uma reconstrução didática independente.

### Materiais complementares

- [Guia Técnico](./reference_materials/15_augmentation_robustez.pdf)
- [Cheat Sheet](./cheat_sheets/cheat_sheet_augmentation_robustez.pdf)


## Como estudar este capítulo

1. Leia a teoria e deduza as expressões apresentadas.
2. Execute as células do **núcleo obrigatório** em um kernel limpo.
3. Confira a interpretação e responda ao exercício antes de abrir a solução.
4. Se tiver TensorFlow e Keras compatíveis, execute o **laboratório opcional**.

**Pré-requisitos transversais:** Python (01), Estatística (03), preparação/validação de ML (05); aplicações textuais remetem à PLN (06). Não há necessidade de repetir integralmente esses cursos.

## Objetivos de aprendizagem

- Escolher transformações plausíveis e impedir vazamento entre imagens relacionadas.
- Relacionar a técnica à sua função de perda, representação e protocolo de avaliação.
- Explicar limitações e formular um teste reprodutível.

## 1. Fundamentação conceitual

Transformações de treino precisam preservar o rótulo. Flips podem ser inadequados em números e medicina; crops podem eliminar o objeto. Em dados com múltiplas imagens por pessoa, dividir por indivíduo antes de augmentar.

**Fonte histórica de apoio:** notas do curso original, consultadas apenas como referência de contexto; as explicações e exemplos desta edição são independentes.

**Questões críticas:** quais pressupostos são necessários? O que a função aprende? Qual métrica indica utilidade real? Que tipo de vazamento ou má interpretação invalidaria a conclusão?

## 2. Matemática essencial

**Expressão esquemática:** `X_aug=T(X_treino)`

A expressão acima resume a relação central. Para a derivação, identifique dimensões, hipóteses e operações; não interprete notação compacta como uma implementação completa.

## 3. Experimento obrigatório — NumPy e/ou scikit-learn

Este experimento é propositalmente pequeno: **executa sem GPU, internet ou serviço externo**. Quando há treino de baseline, não interpretá-lo como evidência da qualidade de uma rede neural.

In [1]:
import numpy as np
im=np.arange(16).reshape(4,4)
flip=np.fliplr(im);noise=np.random.default_rng(7).normal(0,.01,size=im.shape)
print('Original:',im[0].tolist(),'flip horizontal:',flip[0].tolist(),'ruído média:',round(noise.mean(),4))
assert np.array_equal(flip[:,::-1],im)

Original: [0, 1, 2, 3] flip horizontal: [3, 2, 1, 0] ruído média: -0.0008


### Interpretação e limitações

Compare o resultado obtido com a formulação matemática. Confirme shapes, escala e hipótese de modelagem. **Um experimento sintético valida uma operação, não substitui avaliação em dados reais.** Ao treinar modelos, separe dados e ajuste transformações apenas no treino; preserve um conjunto final de teste.

## 4. Laboratório neural opcional — TensorFlow / Keras 3

Esta seção demonstra **APIs reais**, mas depende de versões compatíveis e pode exigir mais recursos que o núcleo. Não é necessária para executar o notebook básico. Não se deve confundir um teste de chamada de API com avaliação experimental de desempenho. Se o pacote TensorFlow não estiver disponível, será exibido um aviso de não execução.

In [2]:
# Laboratório opcional: execute após instalar e configurar TensorFlow/Keras compatíveis.
# Se TensorFlow estiver ausente, o exemplo é ignorado com aviso explícito.
import importlib.util
if importlib.util.find_spec('tensorflow') is None:
    print('Laboratório TensorFlow/Keras não executado: TensorFlow ausente neste kernel.')
else:
    import keras
    from keras import layers
    aug=keras.Sequential([layers.RandomFlip('horizontal'),layers.RandomRotation(.08)])
    print('Camadas de augmentation:',len(aug.layers))

I0000 00:00:1794602046.469579   31076 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Camadas de augmentation: 2


## 5. Exercício de fixação

**Questão:** Por que augmentação deve ocorrer após separar treino e teste?

<details>
<summary>Confira a solução comentada</summary>

**Solução:** Para evitar versões transformadas da mesma imagem aparecendo em conjuntos diferentes.

**Verificação:** explique o resultado usando notação, shapes e consequências para generalização.
</details>

## 6. Erros comuns e boas práticas

- Confundir ajuste no treino com generalização para dados não vistos.
- Reportar somente uma métrica ou uma execução aleatória.
- Ignorar shape, tipo de variável e convenção do backend.
- Executar exemplos com pesos baixados sem registrar versão, licença e procedência.
- Confundir demonstração matemática com validação de um modelo de produção.

## 7. Resumo de consulta rápida

**Ideia central:** Transformações de treino precisam preservar o rótulo.

**Expressão:** `X_aug=T(X_treino)`.

**Lembrete:** começar por baseline; respeitar separação de dados; documentar decisões; avaliar limitações.

**Ponte com outros cursos:** ML I (05) para métricas e validação; PLN (06) para modelos de linguagem; projetos multidisciplinares ficam reservados à fase posterior.

**Fontes de atualização (consultar quando necessário):** [Keras 3](https://keras.io/guides/), [TensorFlow](https://www.tensorflow.org/guide), [scikit-learn](https://scikit-learn.org/stable/user_guide.html).

## Laboratório aplicado revisado — Separação por grupo antes da transformação de treino

Augmentation só pode agir sobre o conjunto de treino, após separação por grupo/indivíduo quando houver dependência.

**Procedimento:** execute a próxima célula com o núcleo declarado em `requirements.txt`, confira os `assert` e interprete os resultados. O experimento é independente dos exemplos anteriores.

In [3]:
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
rng=np.random.default_rng(15);img=rng.normal(size=(24,8,8));groups=np.repeat(np.arange(8),3)
split=GroupShuffleSplit(n_splits=1,test_size=.25,random_state=15)
tr,te=next(split.split(img,groups=groups))
assert not (set(groups[tr]) & set(groups[te]))
aug=img[tr]+rng.normal(0,.01,size=img[tr].shape)
print('Grupos treino:',len(set(groups[tr])),'teste:',len(set(groups[te])),'imagens aumentadas:',aug.shape[0])

Grupos treino: 6 teste: 2 imagens aumentadas: 18


### Interpretação, limites e verificação

Aumentar somente imagens do treino depois da divisão; testar se as transformações preservam o significado da classe.

**Checklist:** conferir entradas e saídas, sementes, inexistência de uso dos dados de teste durante ajuste e distinção entre uma demonstração sintética e desempenho em aplicações reais.